In [1]:
import xml.etree.ElementTree as ET
import pandas as pd

# 1. 파일 설정
xml_file = 'full database.xml'
output_csv = 'drug-target_information/DrugBank_Master_Dataset_Detailed.csv'

# 2. 네임스페이스
ns = {'db': 'http://www.drugbank.ca'}

def extract_detailed_data(xml_path):
    rows = []
    context = ET.iterparse(xml_path, events=('end',))
    
    print(f"[{xml_path}] 상세 데이터 추출 시작...")
    
    for event, elem in context:
        tag = elem.tag.replace(f"{{{ns['db']}}}", "")
        
        if event == 'end' and tag == 'drug':
            # type 속성이 없으면 중첩된 태그이므로 건너뛰기 (루트 레벨 drug만 처리)
            if 'type' not in elem.attrib:
                elem.clear()
                continue
            
            try:
                # --- [Drug Info] ---
                drug_id_elem = elem.find("db:drugbank-id[@primary='true']", ns)
                drug_id = drug_id_elem.text if drug_id_elem is not None else None
                
                name_elem = elem.find("db:name", ns)
                drug_name = name_elem.text if name_elem is not None else None
                
                drug_type = elem.attrib.get('type') # small molecule vs biotech
                
                # 개발 단계(Groups) 가져오기
                groups = [g.text for g in elem.findall("db:groups/db:group", ns)]
                groups_str = "; ".join(groups) if groups else "unknown"
                
                # --- [Target Info] ---
                target_types = {
                    'targets': 'Target', 
                }
                
                has_target = False
                
                for xml_tag, category_name in target_types.items():
                    items = elem.findall(f"db:{xml_tag}/db:{xml_tag[:-1]}", ns)
                    
                    for item in items:
                        polypeptide = item.find("db:polypeptide", ns)
                        
                        if polypeptide is not None:
                            # Gene Name
                            gene_elem = polypeptide.find("db:gene-name", ns)
                            gene_name = gene_elem.text if gene_elem is not None else None
                            
                            # UniProt ID
                            uniprot_id = None
                            for ext in polypeptide.findall("db:external-identifiers/db:external-identifier", ns):
                                res = ext.find("db:resource", ns)
                                if res is not None and res.text == 'UniProtKB':
                                    uniprot_id = ext.find("db:identifier", ns).text
                                    break
                            
                            # Organism (새로 추가)
                            organism_elem = polypeptide.find("db:organism", ns)
                            organism = organism_elem.text if organism_elem is not None else None
                            
                            # Target Name (새로 추가)
                            target_name_elem = polypeptide.find("db:name", ns)
                            target_name = target_name_elem.text if target_name_elem is not None else None
                            
                            # 정보 저장 (snake_case, SMILES/Category 제거, organism/target_name 추가)
                            if gene_name or uniprot_id:
                                rows.append({
                                    'drugbank_id': drug_id,
                                    'drug_name': drug_name,
                                    'drug_groups': groups_str,
                                    'gene_name': gene_name,
                                    'uniprot_id': uniprot_id,
                                    'organism': organism,
                                    'target_name': target_name
                                })
                                has_target = True

            except Exception as e:
                pass
            
            elem.clear()

    return pd.DataFrame(rows)

# 실행
df_detailed = extract_detailed_data(xml_file)

# 결과 확인
print(f"총 데이터 수: {len(df_detailed)}")
print(df_detailed[['drug_name', 'drug_groups', 'gene_name', 'organism']].head())

# 저장
df_detailed.to_csv(output_csv, index=False, encoding='utf-8-sig')
print(f"저장 완료: {output_csv}")

[full database.xml] 상세 데이터 추출 시작...
총 데이터 수: 22820
   drug_name          drug_groups gene_name organism
0  Lepirudin  approved; withdrawn        F2   Humans
1  Cetuximab             approved      EGFR   Humans
2  Cetuximab             approved    FCGR3B   Humans
3  Cetuximab             approved      C1QA   Humans
4  Cetuximab             approved      C1QB   Humans
저장 완료: drug-target_information/DrugBank_Master_Dataset_Detailed.csv


In [2]:
# 전체 데이터 확인
df_detailed

,drugbank_id,drug_name,drug_groups,gene_name,uniprot_id,organism,target_name
0,DB00001,Lepirudin,approved; withdrawn,F2,P00734,Humans,Prothrombin
1,DB00002,Cetuximab,approved,EGFR,P00533,Humans,Epidermal growth factor receptor
2,DB00002,Cetuximab,approved,FCGR3B,O75015,Humans,Low affinity immunoglobulin gamma Fc region re...
3,DB00002,Cetuximab,approved,C1QA,P02745,Humans,Complement C1q subcomponent subunit A
4,DB00002,Cetuximab,approved,C1QB,P02746,Humans,Complement C1q subcomponent subunit B
...,...,...,...,...,...,...,...
22815,DB19353,Benzgalantamine,approved,None,A9X444,Humans,Muscle nicotinic acetylcholine receptor
22816,DB19372,Zoxazolamine,withdrawn,KCNN2,Q9H2S1,Humans,Small conductance calcium-activated potassium ...
22817,DB19378,Megestrol,approved,PGR,P06401,Humans,Progesterone receptor
22818,DB19381,AZACYCLONOL,withdrawn,HRH1,P35367,Humans,Histamine H1 receptor


In [3]:
# === 테이블 분리 ===

# 1. Drug 테이블 (중복 제거)
drug_table = df_detailed[['drugbank_id', 'drug_name', 'drug_groups']].drop_duplicates()
print(f"Drug 테이블: {len(drug_table)}개")
drug_table.head()

Drug 테이블: 9241개


,drugbank_id,drug_name,drug_groups
0,DB00001,Lepirudin,approved; withdrawn
1,DB00002,Cetuximab,approved
9,DB00003,Dornase alfa,approved
10,DB00004,Denileukin diftitox,approved; investigational
12,DB00005,Etanercept,approved; investigational


In [4]:
# 2. Target 테이블 (중복 제거)
target_table = df_detailed[['gene_name', 'uniprot_id', 'organism', 'target_name']].drop_duplicates()
print(f"Target 테이블: {len(target_table)}개")
target_table.head()

Target 테이블: 4917개


,gene_name,uniprot_id,organism,target_name
0,F2,P00734,Humans,Prothrombin
1,EGFR,P00533,Humans,Epidermal growth factor receptor
2,FCGR3B,O75015,Humans,Low affinity immunoglobulin gamma Fc region re...
3,C1QA,P02745,Humans,Complement C1q subcomponent subunit A
4,C1QB,P02746,Humans,Complement C1q subcomponent subunit B


In [5]:
# 3. Drug-Target 관계 테이블 (중복 제거)
drug_target_table = df_detailed[['drugbank_id', 'uniprot_id']].drop_duplicates()
print(f"Drug-Target 관계: {len(drug_target_table)}개")
drug_target_table.head()

Drug-Target 관계: 22697개


,drugbank_id,uniprot_id
0,DB00001,P00734
1,DB00002,P00533
2,DB00002,O75015
3,DB00002,P02745
4,DB00002,P02746


In [10]:
# 4. 저장
drug_table.to_csv('drug-target_information/drug_in_target.csv', index=False, encoding='utf-8-sig')
target_table.to_csv('drug-target_information/target_information.csv', index=False, encoding='utf-8-sig')
drug_target_table.to_csv('drug-target_information/drug_target.csv', index=False, encoding='utf-8-sig')

print("테이블 분리 완료!")
print("- drug_table.csv")
print("- target_table.csv")
print("- drug_target_relation.csv")

테이블 분리 완료!
- drug_table.csv
- target_table.csv
- drug_target_relation.csv


In [7]:
# 통계 확인
print(f"총 Drug-Target 관계: {len(df_detailed)}개")
print(f"고유 약물 수: {len(drug_table)}개")
print(f"고유 타겟 수: {len(target_table)}개")
print(f"고유 관계 수: {len(drug_target_table)}개")

총 Drug-Target 관계: 22820개
고유 약물 수: 9241개
고유 타겟 수: 4917개
고유 관계 수: 22697개


In [8]:
# Organism 분포 확인
target_table['organism'].value_counts().head(10)

organism
Humans                                                                          2981
Escherichia coli (strain K12)                                                    317
Plasmodium falciparum (isolate 3D7)                                              100
Mycobacterium tuberculosis (strain ATCC 25618 / H37Rv)                            74
Bacillus subtilis (strain 168)                                                    66
Thermotoga maritima (strain ATCC 43589 / MSB8 / DSM 3109 / JCM 10099)             55
Thermus thermophilus (strain HB8 / ATCC 27634 / DSM 579)                          47
Pseudomonas aeruginosa (strain ATCC 15692 / PAO1 / 1C / PRS 101 / LMG 12228)      42
Shigella flexneri                                                                 37
Salmonella typhimurium (strain LT2 / SGSC1412 / ATCC 700720)                      33
Name: count, dtype: int64

In [9]:
# Drug groups 분포 확인
drug_table['drug_groups'].value_counts().head(10)

drug_groups
experimental                               4505
investigational                            2044
approved; investigational                  1058
approved                                    889
approved; vet_approved                      114
experimental; investigational                71
approved; investigational; vet_approved      67
approved; withdrawn                          62
approved; investigational; withdrawn         59
approved; experimental                       54
Name: count, dtype: int64